# Interactive self-eval

A two-phase loop over **one persistent game** and **one conversation**:

1. **Player phase.** Ask about the *current scene* — usually "what's the best
   move?", sometimes a general question ("are you facing the gold?"). The
   player answers in a **single generation**: one move token at most, and that
   move is **not applied yet**. It may run `[SEARCH <query>]` memory searches
   (semantic model + past reasoning only, settings-scrubbed) before answering.
2. **Analyst phase.** Control returns to you: the lower text box is pre-filled
   with the default analysis question — edit it or just press **Analyze**. The
   analyst reviews ONLY the player's latest reply, with privileged access (the
   exact frame **and its Settings JSON**), searches all memory tiers if it
   wants, and ends with a rating in [-1.0, 1.0]. Its verdict is stored in the
   same conversation, so the player sees past analyses in later rounds.
   **Only then** is the pending move propagated and the new board shown.

Buttons: **Ask** submits a player question; **Ask & reset game** first swaps in
a brand-new random bare board (recorded in the agent's memory as an explicit
reset) and then submits the question; **Restart conversation** starts a fresh
thread + board.

Prereqs (same as `play.ipynb`):
- Neo4j is up (`bash scripts/vast_neo4j_launch.sh`, or `docker compose up -d neo4j`).
- The semantic model is seeded once (`python -m agent.runner seed`).
- `bash scripts/setup_env.sh` (installs deps incl. `ipywidgets`, and downloads
  the spaCy + GLiNER weights NAMS' extraction pipeline needs).

The first cell connects NAMS and loads Gemma, so it can take a minute.

In [ ]:
import os
import sys

# Run from the repo root so `agent` imports and `.env` resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from agent.self_eval_session import InteractiveSelfEvalSession

# Connects NAMS on a background loop and loads Gemma 4 E4B (first run is slow).
# Logging is on by default: every LLM generate call and every DB retrieval for
# this session is written under session.logger.run_dir.
session = InteractiveSelfEvalSession()
print("session_id:", session.session_id)
if session.logger is not None:
    print("log dir:", session.logger.run_dir)

In [ ]:
import ipywidgets as widgets
from IPython.display import Image, clear_output, display

from agent.self_eval_session import DEFAULT_ANALYST_QUESTION

# ----------------------------------------------------------- player controls
question_box = widgets.Textarea(
    value="",
    placeholder="Ask the player about the current scene (e.g. 'What is the best move?')...",
    description="You:",
    layout=widgets.Layout(width="600px", height="70px"),
)
ask_btn = widgets.Button(description="Ask", button_style="primary")
ask_reset_btn = widgets.Button(description="Ask & reset game", button_style="primary")
restart_btn = widgets.Button(description="Restart conversation", button_style="warning")
dump_btn = widgets.Button(description="Dump DB status", button_style="info")
out = widgets.Output()

# ---------------------------------------------------------- analyst controls
# Pre-filled with the default analysis question; edit it or submit as-is.
# Enabled only while a player reply is awaiting analysis.
analyst_box = widgets.Textarea(
    value=DEFAULT_ANALYST_QUESTION,
    description="Analyst:",
    layout=widgets.Layout(width="600px", height="70px"),
)
analyze_btn = widgets.Button(description="Analyze", button_style="success")

# Rendered frames are 224x224 native; upscale on display so the agent and gold
# are clearly visible.
FRAME_WIDTH = 420  # px


def _sync_phase():
    """Enable exactly the controls of the current phase (no double-asks, no
    analyzing before playing)."""
    player_on = session.phase == "player"
    question_box.disabled = not player_on
    ask_btn.disabled = not player_on
    ask_reset_btn.disabled = not player_on
    analyst_box.disabled = player_on
    analyze_btn.disabled = player_on


def _show_frame(path, caption):
    if path:
        print(caption)
        display(Image(filename=path, width=FRAME_WIDTH))


def _show_searches(searches):
    for s in searches:
        print(f"  [SEARCH {s['query']}]")


def _show_player_result(result):
    _show_frame(result["before_path"], "-- frame the player saw --")
    _show_searches(result.get("searches", []))
    print(f"Player: {result['raw']}")
    if result["action"]:
        print(f"[pending move: {result['action']} -- NOT applied until the analysis is done]")
    else:
        print("[no move token: a prose answer]")
    print("\n>>> Analyst phase: edit or submit the question in the lower box.")


def _show_analyst_step(step):
    """Render each analyst generation as it happens (search calls included)."""
    if step["search_query"] is not None:
        print(f"Analyst: {step['text']}")
        print(f"  [running SEARCH: {step['search_query']}]")
    else:
        print(f"Analyst: {step['text']}")


def _ask_player(q):
    print(f"\n=== You: {q}")
    result = session.ask_player(q)
    _show_player_result(result)
    question_box.value = ""


def on_ask(_):
    q = question_box.value.strip()
    if not q:
        return
    ask_btn.disabled = ask_reset_btn.disabled = True
    try:
        with out:
            _ask_player(q)
    finally:
        _sync_phase()


def on_ask_reset(_):
    q = question_box.value.strip()
    if not q:
        return
    ask_btn.disabled = ask_reset_btn.disabled = True
    try:
        with out:
            session.reset_game()  # recorded in the agent's memory
            print("\n*** Game reset: brand-new random board. ***")
            _ask_player(q)
    finally:
        _sync_phase()


def on_analyze(_):
    analyze_btn.disabled = True
    try:
        with out:
            q = analyst_box.value.strip()
            print(f"\n=== Analyst question: {q or DEFAULT_ANALYST_QUESTION}")
            result = session.ask_analyst(q, on_step=_show_analyst_step)
            if result["action"]:
                print(
                    f"\n[move {result['action']} propagated  "
                    f"gold_collected={result['gold_collected']}  "
                    f"gold_remaining={result['gold_remaining']}]"
                )
                _show_frame(result["frame_path"], "-- board after the propagated move --")
                if result["gold_remaining"] == 0:
                    print("*** Gold collected! Reset the game to keep playing. ***")
            else:
                print("\n[no pending move to propagate]")
            print("\n>>> Player phase: ask the next question in the upper box.")
        analyst_box.value = DEFAULT_ANALYST_QUESTION
    finally:
        _sync_phase()


def on_restart(_):
    info = session.restart()
    with out:
        clear_output()
        print(f"New conversation. session_id={info['session_id']}")
        _show_frame(info["frame_path"], "-- fresh board --")
    analyst_box.value = DEFAULT_ANALYST_QUESTION
    _sync_phase()


def on_dump(_):
    dump_btn.disabled = True
    try:
        with out:
            info = session.dump_db()
            print(f"\nDB dumped -> {info['path']}")
            print(f"  nodes = {info['nodes']}   relationships = {info['relationships']}")
            if session.logger is not None:
                print(f"  llm log: {session.logger.llm_txt}")
                print(f"  db  log: {session.logger.db_txt}")
    finally:
        dump_btn.disabled = False


ask_btn.on_click(on_ask)
ask_reset_btn.on_click(on_ask_reset)
analyze_btn.on_click(on_analyze)
restart_btn.on_click(on_restart)
dump_btn.on_click(on_dump)

display(widgets.VBox([
    question_box,
    widgets.HBox([ask_btn, ask_reset_btn, restart_btn, dump_btn]),
    out,
    analyst_box,
    widgets.HBox([analyze_btn]),
]))

_sync_phase()
# Show the current starting board.
with out:
    print(f"session_id={session.session_id}")
    _show_frame(session.current_frame_path(), "-- starting board --")

### Restore the DB to "semantic seeding only" (optional cleanup)

Same escape hatch as in `play.ipynb`: deletes **all** episodic nodes
(`Conversation`, `Message`, `GameSnapshot`, `ReasoningTrace`/`ReasoningStep`)
and keeps only the seeded semantic model (`Entity` + `Preference` — including
any tips saved via `[WRITE_TIP]`, so wipe those manually if unwanted). Gated by
`if False:` so it never runs by accident.

In [ ]:
# Flip to `if True:` to wipe episodic memory and restore the seeded state.
if False:
    import shutil

    deleted = session.reset_memory_to_seed()
    print("deleted nodes:", deleted)

    img_dir = session.cfg.image_dir
    if img_dir.exists():
        shutil.rmtree(img_dir, ignore_errors=True)
        print("cleared image dir:", img_dir)

    info = session.restart()
    print("new session_id:", info["session_id"])

When you're done, release the model and close the memory client:

In [ ]:
session.close()